# Gauss's 17-gon with ruler and compass

*maths · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/gauss_heptadecagon)

In 1796, a month before turning 19, Gauss found that the regular 17-gon can be drawn with ruler and compass, the first new constructible polygon since the Greeks. The 17th roots of unity, grouped into Gauss periods, solve a chain of quadratics. That chain writes $\cos(2\pi/17)$ with square roots only. Richmond's 1893 construction then draws the polygon step by step, using line and circle intersections written as tensor operations. A grid shows which $n$-gons are constructible (the Gauss–Wantzel theorem). Everything here is proven classical mathematics.

**Vector form**

$$\eta_{m,i} = \sum_{j \equiv i \ (2^m)} \zeta^{g^j},\quad \eta_{m+1,i} + \eta_{m+1,i+2^m} = \eta_{m,i},\quad \eta_{m+1,i}\,\eta_{m+1,i+2^m} = c_i + \sum_k C_{ik}\,\eta_{m,k},\qquad 16\cos\tfrac{2\pi}{17} = -1+\sqrt{17}+\sqrt{34-2\sqrt{17}}+2\sqrt{17+3\sqrt{17}-\sqrt{34-2\sqrt{17}}-2\sqrt{34+2\sqrt{17}}}$$

**Index (tensor) form**

$$\eta_{mi} = M_{mij}\,\zeta_{e_j},\quad M_{mij} = [\,j \equiv i \bmod 2^m\,],\qquad p_i = c_i + C_{ik}\,\eta_{mk},\quad \eta_{m+1,i} = \tfrac12\big(\eta_{mi} \pm \sqrt{\eta_{mi}^2 - 4p_i}\big),\qquad n \text{ constructible} \Leftrightarrow \varphi(n) = 2^r$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\zeta_k = e^{2\pi i k/17}$ | `polar(ones, 2 * pi * k / 17)` | (17,) complex | the 17th roots of unity, the vertices of the 17-gon in the complex plane |
| $e_j = g^j \bmod 17$ | `cyclic_order(g)` | (16,) | the exponents in the order of powers of a primitive root, $g = 3$: 1, 3, 9, 10, 13, … |
| $M_{mij}$ | `(j % 2**m == i) & (i < 2**m)` | (5, 16, 16) | which positions of the cyclic order make up period $i$ at level $m$: every $2^m$-th one |
| $\eta_{mi}$ | `einsum('mij,j->mi', M, zeta[order])` | (5, 16) | repeated $j$ ⇒ summed: all Gauss periods of all five levels (lengths 16, 8, 4, 2, 1) in one go |
| $c_i,\ C_{ik}$ | `one_hot((a + b) % 17).sum()` | (2^m,), (2^m, 2^m) | the product of two sibling periods is a sum of $\zeta^{a+b}$; counting the exponents gives integers that are constant on the parent level's periods |
| $p_i$ | `const + coef @ tower[m, :n]` | (2^m,) | repeated $k$ ⇒ summed: the constant term of each quadratic from the level above |
| $\eta_{m+1,i}$ | `(s + sqrt(s * s - 4 * p)) / 2` | (2^m,) | one square root per level: $16 \to 8 \to 4 \to 2 \to 1$; level 3 holds $2\cos(2\pi k/17)$ |
| $P = A + t(B - A)$ | `line_line, line_circle, circle_circle` | (2,), (2, 2) | the three ruler-and-compass moves as tensor formulas: a $2\times 2$ cross-product solve, a quadratic in $t$, and the radical line of two circles |
| $\varphi(n)$ | `phi[p::p] -= phi[p::p] // p` | (N + 1,) | Euler's totient by a sieve; constructible exactly when $\varphi(n)$ is a power of 2, tested as $x \,\&\, (x-1) = 0$ |
| $n = 2^k \cdot \text{odd}$ | `(2**32 - 1) % (n // (n & -n)) == 0` | (N,) | the odd part must divide $3\cdot5\cdot17\cdot257\cdot65537 = 2^{32}-1$: a product of distinct Fermat primes |

Gauss found the 17-gon on 30 March 1796, just before his 19th birthday, and it reportedly made him choose mathematics over philology. He published the theory in the last section of the Disquisitiones Arithmeticae (1801). The idea: the 16 nontrivial roots $\zeta^k$ are permuted cyclically by $k \mapsto 3k \bmod 17$, because 3 is a primitive root. Summing every second root in this cyclic order gives two periods of 8 terms with $\eta_0 + \eta_1 = -1$ and $\eta_0\eta_1 = -4$, so $\eta_{0,1} = (-1 \pm \sqrt{17})/2$. Halving again gives periods of 4, 2 and 1 terms. Each pair is the root pair of a quadratic whose coefficients are periods one level up, because the subgroups have order $16 \to 8 \to 4 \to 2 \to 1$, all powers of 2. The 2-term periods are $\zeta^k + \zeta^{-k} = 2\cos(2\pi k/17)$, so four square roots give Gauss's formula. The program checks every quadratic relation with exact integer coefficients, rebuilds every period from square roots alone, and compares the closed formula with `torch.cos` (and with 60-digit sympy when it is installed). Any primitive root (3, 5, 6, 7, 10, 11, 12, 14) gives the same tree in a different order. A square root is a ruler-and-compass operation, so the polygon is constructible. H. W. Richmond (1893) gave a short recipe on the unit circle. Take $OJ = OB/4$ on the vertical radius. Put $E$ on the diameter with $\angle OJE = \tfrac14\angle OJP_0$ (two angle bisections), and $F$ with $\angle EJF = 45°$. The circle on the diameter $FP_0$ meets $OB$ at $K$. The circle about $E$ through $K$ cuts the diameter at $N_3 = \cos(6\pi/17)$ and $N_5 = \cos(10\pi/17)$, and perpendiculars there give $P_3$ and $P_5$. Here every step really draws a line through two points or a circle about a point through another. Each new point is a line–line, line–circle or circle–circle intersection, and the remaining vertices are stepped off with circles: about $P_k$ through $P_{k-1}$, and about $P_0$ through $P_k$ for the mirror image. The 55 steps put all 17 vertices within about $10^{-15}$ of $e^{2\pi i k/17}$ in float64; switch to float32 to see the rounding grow to about $10^{-6}$. Gauss stated that the $n$-gon is constructible when $n = 2^k p_1 \cdots p_r$ with distinct Fermat primes $p_i = 2^{2^j} + 1$. Wantzel (1837) proved that no other $n$ works, which also settled angle trisection and cube doubling. This is equivalent to Euler's $\varphi(n)$ being a power of 2, and the grid checks both tests up to $n$. The only known Fermat primes are 3, 5, 17, 257 and 65537. Richelot published the 257-gon in 1832, and J. G. Hermes spent about ten years on the 65537-gon, finished in 1894 and kept in Göttingen. Gauss reportedly asked for a 17-gon on his tombstone; the stonemason is said to have refused because it would look like a circle. The Gauss monument in Braunschweig carries a 17-pointed star instead.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/gauss_heptadecagon/main.py
"""Main loop: parameters → roots of unity and Gauss periods → nested radical → Richmond's construction → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    dtype = torch.float64 if params["precision"] == "float64" else torch.float32
    g = int(params["generator"])
    n_max = params["n_max"]

    # 1. roots of unity and the period tree, all five levels in one einsum
    zeta = tensors.roots_of_unity(17, torch.float64, dev)  # (17,) complex128
    order = tensors.cyclic_order(g)
    print(f"17th roots of unity: |Σ ζ^k| = {zeta.sum().abs():.1e}")
    print(f"generator g = {g}: exponents g^j mod 17 = {order.tolist()}")
    eta = equation.periods(zeta, order)  # (5, 16)
    worst = 0.0
    for m in range(4):
        n = 2**m
        const, coef = equation.product_in_periods(order, m)
        a, b = eta[m + 1, :n], eta[m + 1, n : 2 * n]
        prod = const.to(eta.dtype) + coef.to(eta.dtype) @ eta[m, :n]
        worst = max(worst, (a + b - eta[m, :n]).abs().max().item(), (a * b - prod).abs().max().item())
        terms = [f"{int(c) if c != 1 else ''}η{m},{k}" for k, c in enumerate(coef[0].tolist()) if c]
        terms = " + ".join(([str(int(const[0]))] if const[0] else []) + terms)
        print(
            f"level {m + 1}: {2 * n:>2} periods of {16 // (2 * n):>2} terms, e.g. "
            f"η{m + 1},0 + η{m + 1},{n} = η{m},0 and η{m + 1},0·η{m + 1},{n} = {terms}"
        )
    print(f"  all 15 quadratic relations hold to {worst:.1e}")
    print(f"  η1,0 = {eta[1, 0].real:.12f} = (−1 + √17)/2 = {(-1 + math.sqrt(17)) / 2:.12f}")

    # 2. the tower by square roots, and Gauss's closed formula
    tower = solve.periods_by_radicals(eta, order)
    print(f"tower of square roots reproduces every period to {(tower - eta).abs().max():.1e}")
    gauss = equation.gauss_formula(torch.float64)
    exact = math.cos(2 * math.pi / 17)
    print(
        f"Gauss: 16 cos(2π/17) = {16 * gauss:.15f}; cos(2π/17) = {gauss:.16f}, torch.cos error {abs(gauss - exact):.1e}"
    )
    try:
        import sympy as sp

        s17 = sp.sqrt(17)
        expr = (
            -1
            + s17
            + sp.sqrt(34 - 2 * s17)
            + 2 * sp.sqrt(17 + 3 * s17 - sp.sqrt(34 - 2 * s17) - 2 * sp.sqrt(34 + 2 * s17))
        ) / 16
        print(f"  sympy, 60 digits: |formula − cos(2π/17)| = {float(abs(sp.N(expr - sp.cos(2 * sp.pi / 17), 60))):.1e}")
    except ImportError:
        print("  (sympy not installed: symbolic check skipped)")

    # 3. Richmond's construction
    con = solve.richmond(dtype, dev)
    k = torch.arange(17, dtype=torch.float64, device=dev)
    exact_v = torch.stack([torch.cos(2 * math.pi * k / 17), torch.sin(2 * math.pi * k / 17)], 1)
    err = (con["vertices"].double() - exact_v).norm(dim=1)
    n_circ = sum(s["kind"] == "circle" for s in con["steps"])
    pts = con["points"]
    print(
        f"Richmond (1893) in {params['precision']}: {len(con['steps'])} steps "
        f"({n_circ} circles, {len(con['steps']) - n_circ} lines)"
    )
    print(f"  N3 = {pts['N3'][0]:.15f}  vs cos(6π/17)  = {math.cos(6 * math.pi / 17):.15f}")
    print(f"  N5 = {pts['N5'][0]:.15f} vs cos(10π/17) = {math.cos(10 * math.pi / 17):.15f}")
    closure = (pts["P0'"] - pts["P0"]).norm()
    print(f"  17 vertices: max error {err.max():.1e}; stepping back from P1 lands on P0 within {closure:.1e}")

    # 4. Gauss–Wantzel
    grid = solve.constructibility(n_max, dev)
    yes = grid["n"][grid["fermat"]]
    agree = bool((grid["phi_test"] == grid["fermat"]).all())
    print(f"constructible n-gons, n ≤ {n_max}: {len(yes)} of {n_max - 2}; φ(n) a power of 2 agrees: {agree}")
    print("  " + ", ".join(str(int(v)) for v in yes))
    print("  257-gon: Richelot (1832); 65537-gon: Hermes (about ten years' work, finished 1894)")

    data = {
        "con": con,
        "err": err,
        "eta": eta,
        "tower": tower,
        "order": order,
        "gauss": float(gauss),
        "grid": grid,
        "precision": params["precision"],
    }
    return plot.animate(data, out_dir / "gauss_heptadecagon.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/gauss_heptadecagon/equation.py
r"""Mathematics → tensor translation: Gauss periods, the nested radical, ruler-and-compass geometry, Gauss–Wantzel.

1. Gauss periods. With a primitive root g mod 17 the exponents are ordered e_j = g^j mod 17, j = 0 … 15. At level m
   (m = 0 … 4) the 2^m periods of length 16 / 2^m are

        η_{m,i} = Σ_j M_{mij} ζ^{e_j},      M_{mij} = [j ≡ i mod 2^m]      (summed index: j)

   so a whole level is one masked sum over the cyclic order, and all five levels are one einsum.

2. The quadratics. Period i of level m splits into the two periods i and i + 2^m of level m + 1, with
        sum      η_{m+1,i} + η_{m+1,i+2^m} = η_{m,i}
        product  η_{m+1,i} η_{m+1,i+2^m}   = c_i + Σ_k C_{ik} η_{m,k}       (summed index: k)
   The product is a sum of ζ^{a+b} over the exponents a, b of the two halves; counting how often each a + b mod 17
   occurs gives integers that are constant on the level-m periods (that is Gauss's key lemma). So every η of level
   m + 1 is a root of t² − s t + p = 0 with s, p known from level m: one square root per level.

3. Ruler and compass. A construction step draws a line through two points or a circle about a point through
   another; new points are line–line, line–circle and circle–circle intersections, written below as tensor ops.

4. Gauss–Wantzel. The regular n-gon is constructible iff n = 2^k p₁ ⋯ p_r with distinct Fermat primes p_i, iff
   Euler's φ(n) is a power of 2.
"""

import torch

from .tensors import FERMAT_PRIMES, P, level_masks

# ---------------------------------------------------------------- 1–2. periods and their quadratics


def periods(zeta: torch.Tensor, order: torch.Tensor) -> torch.Tensor:
    """η_{m,i} = Σ_j M_{mij} ζ^{e_j} for all five levels at once.   (17,), (16,) → (5, 16) complex, padded with 0"""
    masks = level_masks(zeta.device).to(zeta.dtype)  # (5, 16, 16)
    return torch.einsum("mij,j->mi", masks, zeta[order.to(zeta.device)])


def product_in_periods(order: torch.Tensor, m: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Exact integer coefficients of η_{m+1,i} η_{m+1,i+2^m} = c_i + Σ_k C_{ik} η_{m,k}.   → (2^m,), (2^m, 2^m)"""
    n = 2**m
    exps = order.reshape(-1, 2 * n).T  # (2n, L): row r = exponents of period r at level m + 1
    a, b = exps[:n], exps[n:]  # the two halves of every level-m period
    sums = (a[:, :, None] + b[:, None, :]) % P  # (n, L, L) exponent of each term ζ^a ζ^b
    counts = torch.nn.functional.one_hot(sums, P).sum(dim=(1, 2))  # (n, 17): how often each exponent occurs
    on_periods = counts[:, order.reshape(-1, n)]  # (n, 16/n, n): count at every exponent of every level-m period
    if not (on_periods == on_periods[:, :1]).all():
        raise ArithmeticError("product is not a combination of level-m periods")
    return counts[:, 0], on_periods[:, 0]


def solve_quadratic(s: torch.Tensor, p: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Both roots of t² − s t + p = 0, with one complex square root: t = (s ± √(s² − 4p)) / 2."""
    root = torch.sqrt(s * s - 4 * p)
    return (s + root) / 2, (s - root) / 2


def gauss_formula(dtype: torch.dtype) -> torch.Tensor:
    """Gauss (1796): 16 cos(2π/17) = −1 + √17 + √(34 − 2√17) + 2√(17 + 3√17 − √(34 − 2√17) − 2√(34 + 2√17))."""
    s17 = torch.sqrt(torch.tensor(17.0, dtype=dtype))
    a = torch.sqrt(34 - 2 * s17)
    b = torch.sqrt(34 + 2 * s17)
    return (-1 + s17 + a + 2 * torch.sqrt(17 + 3 * s17 - a - 2 * b)) / 16


# ---------------------------------------------------------------- 3. ruler and compass


def _cross(u: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    return u[..., 0] * v[..., 1] - u[..., 1] * v[..., 0]


def line_line(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor, d: torch.Tensor) -> torch.Tensor:
    """Intersection of line ab with line cd: a + t(b − a) with t = (c − a) × (d − c) / (b − a) × (d − c).   → (2,)"""
    t = _cross(c - a, d - c) / _cross(b - a, d - c)
    return a + t * (b - a)


def line_circle(a: torch.Tensor, b: torch.Tensor, centre: torch.Tensor, r) -> torch.Tensor:
    """Both points a + t(b − a) at distance r from the centre: |f + t d|² = r² with d = b − a, f = a − centre.

    Rows sorted by t (first the one nearer a's side).   → (2, 2)"""
    d, f = b - a, a - centre
    qa, qb, qc = d @ d, 2 * (f @ d), f @ f - r**2
    root = torch.sqrt(qb * qb - 4 * qa * qc)
    t = torch.stack([(-qb - root), (-qb + root)]) / (2 * qa)  # (2,)
    return a + t[:, None] * d


def circle_circle(c1: torch.Tensor, r1, c2: torch.Tensor, r2) -> torch.Tensor:
    """Both intersections of two circles: along the centre line at distance a from c1, then ± h across it.

    Row 0 lies to the left of the direction c1 → c2, row 1 to the right.   → (2, 2)"""
    d = (c2 - c1).norm()
    e = (c2 - c1) / d  # unit vector c1 → c2
    a = (r1**2 - r2**2 + d**2) / (2 * d)
    h = torch.sqrt(r1**2 - a**2)
    perp = torch.stack([-e[1], e[0]])  # e rotated by +90°
    return c1 + a * e + torch.stack([h, -h])[:, None] * perp


# ---------------------------------------------------------------- 4. Gauss–Wantzel


def euler_phi(n_max: int, device: torch.device | None = None) -> torch.Tensor:
    """φ(n) for n = 0 … n_max by a sieve: for every prime p, φ(n) ← φ(n)(1 − 1/p) on its multiples.   → (n_max+1,)"""
    phi = torch.arange(n_max + 1, device=device)
    for p in range(2, n_max + 1):
        if phi[p] == p:  # untouched so far ⇒ p is prime
            phi[p::p] -= phi[p::p] // p
    return phi


def is_power_of_two(x: torch.Tensor) -> torch.Tensor:
    """x > 0 with a single bit set: x & (x − 1) = 0."""
    return (x > 0) & ((x & (x - 1)) == 0)


def fermat_factorisation(n: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """n = 2^k · (odd part). The n-gon is constructible iff the odd part divides 3·5·17·257·65537 = 2^32 − 1, i.e. it
    is a product of distinct Fermat primes.   (N,) → 2^k (N,), Fermat primes dividing n (N, 5), constructible (N,)"""
    two_power = n & -n  # lowest set bit
    odd = n // two_power
    fermat = torch.tensor(FERMAT_PRIMES, device=n.device)
    divides = odd[:, None] % fermat[None, :] == 0  # (N, 5)
    constructible = ((2**32 - 1) % odd == 0) & (n >= 3)
    return two_power, divides, constructible


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/gauss_heptadecagon/tensors.py
"""Build the concrete tensors: the 17th roots of unity, the cyclic ordering of the exponents and the level masks."""

import math

import torch

P = 17  # a Fermat prime: 17 = 2^(2^2) + 1, so (Z/17)^× has order 16 = 2^4
LEVELS = 5  # period lengths 16, 8, 4, 2, 1
FERMAT_PRIMES = (3, 5, 17, 257, 65537)  # the only known ones; their product is 2^32 − 1


def is_primitive_root(g: int, p: int = P) -> bool:
    """g generates (Z/p)^×: its powers g^0 … g^(p−2) are all different."""
    return len({pow(g, j, p) for j in range(p - 1)}) == p - 1


def cyclic_order(g: int, p: int = P) -> torch.Tensor:
    """The exponents in the order g^0, g^1, …, g^15 mod 17 (Gauss's ordering).   → (16,) int64"""
    if not is_primitive_root(g, p):
        raise ValueError(f"{g} is not a primitive root mod {p}")
    j = torch.arange(p - 1)
    return torch.tensor([pow(g, int(k), p) for k in j])


def roots_of_unity(p: int, dtype: torch.dtype, device: torch.device) -> torch.Tensor:
    """ζ^k = e^{2πik/p} for k = 0 … p − 1.   → (p,) complex"""
    k = torch.arange(p, dtype=dtype, device=device)
    return torch.polar(torch.ones_like(k), 2 * math.pi * k / p)


def level_masks(device: torch.device) -> torch.Tensor:
    """M[m, i, j] = 1 when position j of the cyclic order belongs to period i at level m, i.e. j ≡ i mod 2^m.

    Level m has 2^m periods of 16 / 2^m terms; rows i ≥ 2^m are padding (all zero).   → (5, 16, 16)"""
    m = torch.arange(LEVELS, device=device)[:, None, None]
    i = torch.arange(P - 1, device=device)[None, :, None]
    j = torch.arange(P - 1, device=device)[None, None, :]
    return ((j % 2**m == i) & (i < 2**m)).double()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/gauss_heptadecagon/solve.py
"""Solve the period tower by square roots, run Richmond's construction step by step, and test constructibility."""

import torch

from .equation import (
    circle_circle,
    euler_phi,
    fermat_factorisation,
    is_power_of_two,
    line_circle,
    line_line,
    product_in_periods,
    solve_quadratic,
)

# ---------------------------------------------------------------- the period tower by radicals


def periods_by_radicals(eta: torch.Tensor, order: torch.Tensor) -> torch.Tensor:
    """Rebuild every period from η_{0,0} = −1 by one quadratic per level: t² − η_{m,i} t + (c_i + C_{ik} η_{m,k}) = 0.

    Only square roots are used; the true periods `eta` only decide which root is which (as Gauss decided by
    comparing the sizes of cosines).   (5, 16) complex → (5, 16) complex"""
    tower = torch.zeros_like(eta)
    tower[0, 0] = -1
    for m in range(4):  # 4 levels, each batched over its 2^m quadratics
        n = 2**m
        const, coef = product_in_periods(order, m)
        s = tower[m, :n]  # (n,) sums
        p = const.to(eta.dtype) + coef.to(eta.dtype) @ tower[m, :n]  # (n,) products, repeated k ⇒ summed
        plus, minus = solve_quadratic(s, p)
        first = torch.where((plus - eta[m + 1, :n]).abs() < (minus - eta[m + 1, :n]).abs(), plus, minus)
        tower[m + 1, :n] = first
        tower[m + 1, n : 2 * n] = s - first
    return tower


# ---------------------------------------------------------------- Richmond's construction (1893)


class Construction:
    """A ruler-and-compass log: each step draws one circle (centre through a point) or one line (through two points)."""

    def __init__(self, points: dict):
        self.points = dict(points)
        self.steps: list[dict] = []

    def circle(self, centre: str, through: str, label: str) -> tuple[torch.Tensor, torch.Tensor]:
        c = self.points[centre]
        r = (self.points[through] - c).norm()
        self.steps.append({"kind": "circle", "centre": c, "radius": r, "label": label, "new": []})
        return c, r

    def line(self, a: str, b: str, label: str) -> tuple[torch.Tensor, torch.Tensor]:
        pa, pb = self.points[a], self.points[b]
        self.steps.append({"kind": "line", "a": pa, "b": pb, "label": label, "new": []})
        return pa, pb

    def mark(self, name: str, point: torch.Tensor) -> torch.Tensor:
        self.points[name] = point
        self.steps[-1]["new"].append(name)
        return point


def _far(pts: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
    """Of two candidate points (2, 2), the one farther from ref."""
    return pts[(pts - ref).norm(dim=1).argmax()]


def _perpendicular_foot_vertex(con: Construction, foot: str, vertex: str, gamma) -> None:
    """Perpendicular to the diameter at `foot`, where it meets the circle Γ above the axis is `vertex` (4 steps)."""
    o, x_axis = con.points["O"], (con.points["A"], con.points["P0"])
    c, r = con.circle(foot, "O", f"circle about {foot} through O")
    con.mark(f"S_{foot}", _far(line_circle(*x_axis, c, r), o))  # the mirror image of O in the foot
    c1, r1 = con.circle("O", f"S_{foot}", f"circle about O through S_{foot}")
    c2, r2 = con.circle(f"S_{foot}", "O", f"circle about S_{foot} through O")
    up, down = circle_circle(c1, r1, c2, r2)
    con.points[f"U_{foot}"], con.points[f"D_{foot}"] = up, down
    a, b = con.line(f"U_{foot}", f"D_{foot}", f"perpendicular at {foot}")
    pts = line_circle(a, b, *gamma)
    con.mark(vertex, pts[pts[:, 1].argmax()])


def richmond(dtype: torch.dtype, device: torch.device) -> dict:
    """H. W. Richmond's construction of the regular 17-gon in the unit circle, with P0 = (1, 0).

    J = OB/4 on the vertical radius; E on the diameter with ∠OJE = ∠OJP0 / 4; F with ∠EJF = 45°; the circle on FP0
    meets OB at K; the circle about E through K meets the diameter at N3 = cos(6π/17) and N5 = cos(10π/17); the
    perpendiculars there give P3 and P5. Every point comes from one of the three intersection functions."""
    z = torch.zeros(2, dtype=dtype, device=device)
    con = Construction({"O": z, "P0": torch.tensor([1.0, 0.0], dtype=dtype, device=device)})
    gamma = con.circle("O", "P0", "the circle Γ")
    x_axis = con.line("O", "P0", "diameter through P0")
    con.mark("A", line_circle(*x_axis, *gamma)[0])

    # perpendicular diameter: equal circles about A and P0
    up, down = circle_circle(
        *con.circle("A", "P0", "circle about A through P0"), *con.circle("P0", "A", "circle about P0 through A")
    )
    con.points["X1"], con.points["X2"] = up, down
    y_axis = con.line("X1", "X2", "perpendicular diameter")
    pts = line_circle(*y_axis, *gamma)
    con.mark("B", pts[pts[:, 1].argmax()])

    # J = OB/4 by halving twice
    c, r = con.circle("B", "O", "circle about B through O")
    con.points["Y1"], con.points["Y2"] = circle_circle(*gamma, c, r)
    con.mark("M", line_line(*con.line("Y1", "Y2", "halve OB"), *y_axis))
    pts = circle_circle(
        *con.circle("M", "O", "circle about M through O"), *con.circle("O", "M", "circle about O through M")
    )
    con.points["Z1"], con.points["Z2"] = pts
    con.mark("J", line_line(*con.line("Z1", "Z2", "halve OM: OJ = OB/4"), *y_axis))

    # E: quarter the angle OJP0 by bisecting twice
    j = con.points["J"]
    ray = con.line("J", "P0", "line JP0")
    small = con.circle("J", "O", "circle about J through O")
    con.mark("Q", line_circle(*ray, *small)[1])
    for name, target in (("Q", "R"), ("Q2", "R2")):
        pts = circle_circle(
            *con.circle("O", name, f"circle about O through {name}"),
            *con.circle(name, "O", f"circle about {name} through O"),
        )
        con.points[target] = _far(pts, j)
        bisector = con.line("J", target, "bisect the angle at J" if target == "R" else "bisect again: ∠OJE = ∠OJP0/4")
        if target == "R":
            pts = line_circle(*bisector, *small)
            con.mark("Q2", pts[pts[:, 1].argmin()])
    con.mark("E", line_line(*bisector, *x_axis))

    # F: ∠EJF = 45°, half of a right angle at J
    c, r = con.circle("J", "E", "circle about J through E")
    con.mark("E'", line_circle(con.points["E"], j, c, r)[1])  # the other end of the diameter through E
    pts = circle_circle(
        *con.circle("E", "E'", "circle about E through E'"), *con.circle("E'", "E", "circle about E' through E")
    )
    con.points["W1"], con.points["W2"] = pts
    perp = con.line("W1", "W2", "perpendicular to JE at J")
    pts = line_circle(*perp, c, r)
    con.points["W"] = pts[pts[:, 0].argmin()]  # on the circle about J, left of the vertical
    pts = circle_circle(
        *con.circle("E", "W", "circle about E through W"), *con.circle("W", "E", "circle about W through E")
    )
    con.points["V1"], con.points["V2"] = pts
    con.mark("F", line_line(*con.line("V1", "V2", "bisect: ∠EJF = 45°"), *x_axis))

    # K: the circle on the diameter FP0 meets OB
    pts = circle_circle(
        *con.circle("F", "P0", "circle about F through P0"), *con.circle("P0", "F", "circle about P0 through F")
    )
    con.points["G1"], con.points["G2"] = pts
    con.mark("Mf", line_line(*con.line("G1", "G2", "midpoint of FP0"), *x_axis))
    pts = line_circle(*y_axis, *con.circle("Mf", "P0", "circle on the diameter FP0"))
    con.mark("K", pts[pts[:, 1].argmax()])

    # N3, N5: the circle about E through K meets the diameter
    n5, n3 = line_circle(*x_axis, *con.circle("E", "K", "circle about E through K"))  # sorted along O → P0
    con.mark("N5", n5)
    con.mark("N3", n3)
    _perpendicular_foot_vertex(con, "N3", "P3", gamma)
    _perpendicular_foot_vertex(con, "N5", "P5", gamma)

    # P4 halves the arc P3P5
    pts = circle_circle(
        *con.circle("P3", "P5", "circle about P3 through P5"), *con.circle("P5", "P3", "circle about P5 through P3")
    )
    con.points["H1"], con.points["H2"] = pts
    pts = line_circle(*con.line("H1", "H2", "bisect the arc P3P5"), *gamma)
    con.mark("P4", pts[(pts - con.points["P3"]).norm(dim=1).argmin()])

    # step the side around: the circle about P_k through P_(k−1) meets Γ again at P_(k+1)
    def step(centre: int, back: int, new: int, label: str = "") -> None:
        c, r = con.circle(f"P{centre}", f"P{back}", label or f"circle about P{centre} through P{back}")
        name = f"P{new}" if f"P{new}" not in con.points else f"P{new}'"
        con.mark(name, _far(circle_circle(*gamma, c, r), con.points[f"P{back}"]))

    for k in (5, 6, 7):
        step(k, k - 1, k + 1)  # P6, P7, P8
    for k in (3, 2, 1):
        step(k, k + 1, k - 1)  # P2, P1 and back to P0 (a check)
    for k in range(1, 9):  # the circle about P0 through P_k meets Γ again at the mirror image P_(17−k)
        step(0, k, 17 - k, f"circle about P0 through P{k}: mirror P{17 - k}")

    vertices = torch.stack([con.points[f"P{k}"] for k in range(17)])
    return {"points": con.points, "steps": con.steps, "vertices": vertices, "gamma": gamma}


# ---------------------------------------------------------------- Gauss–Wantzel


def constructibility(n_max: int, device: torch.device | None = None) -> dict:
    """Which regular n-gons, 1 ≤ n ≤ n_max, are constructible, two ways at once.   → dict of (N,) tensors"""
    n = torch.arange(1, n_max + 1, device=device)
    phi = euler_phi(n_max, device)[1:]
    two_power, divides, fermat = fermat_factorisation(n)
    return {
        "n": n,
        "phi": phi,
        "phi_test": is_power_of_two(phi) & (n >= 3),
        "fermat": fermat,
        "two_power": two_power,
        "divides": divides,
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/gauss_heptadecagon/plot.py
"""Richmond's construction drawing itself, the Gauss period tree with its square roots, and the Gauss–Wantzel grid."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Circle, Polygon

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import product_in_periods

PER_STEP = 0.8  # frames per construction step (a step fades in over 1.25 frames)
POLY_FRAMES = 10
HOLD = 5
LABELLED = ["O", "P0", "B", "J", "E", "F", "K", "N3", "N5", "P3", "P4", "P5"]
FAINT = "#cbd5e1"


def _bitrev4(i: int) -> int:
    return int(f"{i:04b}"[::-1], 2)


def animate(d: dict, path: Path) -> Path:
    con, steps = d["con"], d["con"]["steps"]
    n_steps = len(steps)
    f_build = round(PER_STEP * n_steps)
    n_frames = f_build + POLY_FRAMES + HOLD

    fig = plt.figure(figsize=(10.4, 5.85))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 0.95], height_ratios=[1.25, 1.0])
    ax_c = fig.add_subplot(gs[:, 0])
    ax_t = fig.add_subplot(gs[0, 1])
    ax_g = fig.add_subplot(gs[1, 1])

    # ---------------------------------------------------------------- the construction
    lim = 1.32
    ax_c.set_xlim(-lim, lim)
    ax_c.set_ylim(-lim, lim)
    ax_c.set_aspect("equal")
    ax_c.set_xticks([])
    ax_c.set_yticks([])
    ax_c.grid(False)
    artists = []
    for s in steps:
        if s["kind"] == "circle":
            c = s["centre"].double().cpu().numpy()
            art = Circle(c, float(s["radius"]), fill=False, lw=1.0, color=ACCENT[0], alpha=0)
            ax_c.add_patch(art)
        else:
            a, b = s["a"].double().cpu().numpy(), s["b"].double().cpu().numpy()
            u = (b - a) / np.linalg.norm(b - a)
            ends = np.stack([a - 4 * u, a + 4 * u])
            (art,) = ax_c.plot(ends[:, 0], ends[:, 1], color="#374151", lw=1.0, alpha=0)
        artists.append(art)
    pts = {k: v.double().cpu().numpy() for k, v in con["points"].items()}
    born = {name: i for i, s in enumerate(steps) for name in s["new"]}
    born.setdefault("O", -1)
    born.setdefault("P0", -1)
    dots = {name: ax_c.plot(*pts[name], "o", ms=3, color="#111827", alpha=0)[0] for name in born}
    tags = {
        name: ax_c.annotate(
            name.replace("P", "$P_{") + "}$" if name[0] == "P" else name,
            pts[name],
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
            alpha=0,
        )
        for name in LABELLED
    }
    verts = d["con"]["vertices"].double().cpu().numpy()
    vdots = ax_c.scatter(verts[:, 0], verts[:, 1], s=28, color=ACCENT[2], zorder=5, alpha=0)
    closed = np.vstack([verts, verts[:1]])
    (edges,) = ax_c.plot([], [], color=ACCENT[2], lw=2.2, zorder=4)
    fill = Polygon(verts, closed=True, color=ACCENT[2], alpha=0, lw=0, zorder=3)
    ax_c.add_patch(fill)
    title_c = ax_c.set_title("", fontsize=10)
    foot = ax_c.text(0.02, 0.02, "", transform=ax_c.transAxes, fontsize=8.5, color="#374151")

    # ---------------------------------------------------------------- the period tree and its square roots
    eta = d["tower"].cpu()
    order = d["order"].cpu()
    ax_t.set_xlim(-0.8, 15.8)
    ax_t.set_ylim(-9.6, 0.6)
    ax_t.axis("off")
    xs = {}
    for m in range(5):
        n = 2**m
        for i in range(n):
            leaves = [_bitrev4(j) for j in range(16) if j % n == i]
            xs[m, i] = float(np.mean(leaves))
    tree_art = {m: [] for m in range(5)}
    for m in range(5):
        for i in range(2**m):
            on_path = i == 0
            colour = ACCENT[1] if on_path else "#111827"
            if m < 4:
                label = f"{eta[m, i].real.item():.2f}"
            else:
                label = f"$\\zeta^{{{int(order[i])}}}$"
            size = 8 if m < 3 else 6.5
            t = ax_t.text(xs[m, i], -m * 1.15, label, ha="center", va="center", fontsize=size, color=colour, alpha=0)
            tree_art[m].append(t)
            if m:
                parent = i % 2 ** (m - 1)
                (ln,) = ax_t.plot(
                    [xs[m - 1, parent], xs[m, i]],
                    [-(m - 1) * 1.15 - 0.3, -m * 1.15 + 0.3],
                    color=ACCENT[1] if on_path else FAINT,
                    lw=1,
                    alpha=0,
                )
                tree_art[m].append(ln)
    for m, name in enumerate(["16 terms", "8 + 8", "4 × 4", "8 × 2: $2\\cos$", "16 × 1"]):
        ax_t.text(-0.7, -m * 1.15 - 0.45, name, fontsize=6.5, color="#6b7280")
    g = int(order[1])
    ax_t.set_title(
        rf"Gauss periods $\eta_{{m,i}} = \sum_j M_{{mij}}\,\zeta^{{{g}^j}}$, halved level by level", fontsize=10
    )
    lines = []
    for m in range(4):
        const, coef = product_in_periods(order, m)
        s = eta[m, 0]
        p = float(const[0]) + (coef[0].double() @ eta[m, : 2**m].real).item()
        r = eta[m + 1, 0]
        val = f"{r.real.item():.6f}" if m < 3 else f"{r.real.item():.4f} + {r.imag.item():.4f}i"
        lines.append(f"$t^2 - ({s.real.item():.4f})\\,t + ({p:.4f}) = 0$  →  $\\eta_{{{m + 1},0}}$ = {val}")
    radical = (
        r"$16\cos\frac{2\pi}{17} = -1+\sqrt{17}+\sqrt{34-2\sqrt{17}}"
        r"+2\sqrt{17+3\sqrt{17}-\sqrt{34-2\sqrt{17}}-2\sqrt{34+2\sqrt{17}}}$"
    )
    rad_art = [ax_t.text(-0.5, -6.0 - 0.75 * m, line, fontsize=8, alpha=0) for m, line in enumerate(lines)]
    rad_art.append(ax_t.text(-0.5, -9.1, radical, fontsize=8.5, alpha=0, color=ACCENT[1]))
    gauss_txt = ax_t.text(15.7, -9.55, "", fontsize=7.5, ha="right", va="top", color="#374151")

    # ---------------------------------------------------------------- Gauss–Wantzel grid
    grid = d["grid"]
    n_all = grid["n"].cpu().numpy()
    yes = grid["fermat"].cpu().numpy()
    n_max = len(n_all)
    cols = 10
    rows = -(-n_max // cols)
    code = np.where(yes, 2, 1)
    code[np.isin(n_all, [3, 5, 17, 257])] = 3
    code[n_all < 3] = 1
    full = np.zeros(rows * cols, dtype=int)
    cmap = ListedColormap(["#ffffff", "#f3f4f6", "#93c5fd", ACCENT[1]])
    img = np.zeros((rows, cols))
    im = ax_g.imshow(img, cmap=cmap, vmin=0, vmax=3, interpolation="nearest", aspect="auto")
    ax_g.set_xticks([])
    ax_g.set_yticks([])
    ax_g.grid(False)
    texts = []
    if rows <= 20:
        for idx, n in enumerate(n_all):
            texts.append(
                ax_g.text(
                    idx % cols,
                    idx // cols,
                    str(n),
                    ha="center",
                    va="center",
                    fontsize=7 if rows <= 10 else 5.5,
                    alpha=0,
                    color="#ffffff" if code[idx] == 3 else "#111827",
                    fontweight="bold" if code[idx] >= 2 else "normal",
                )
            )
    ax_g.set_title(
        f"Gauss–Wantzel: {int(yes.sum())} constructible n-gons for n ≤ {n_max} (red: Fermat primes)", fontsize=10
    )
    ax_g.set_xlabel(
        "$n = 2^k p_1\\cdots p_r$ with distinct Fermat primes $\\Leftrightarrow$ $\\varphi(n)$ a power of 2",
        fontsize=8.5,
    )

    err = d["err"].max().item()

    def update(f: int):
        u = min(f / PER_STEP, n_steps)  # construction progress in steps
        frac = min(1.0, f / f_build)
        for i, art in enumerate(artists):
            age = u - i
            if age <= 0:
                art.set_alpha(0)
                continue
            fresh = age < 2 and f < f_build
            art.set_alpha(min(1.0, age) * (1.0 if fresh else 0.25))
            art.set_linewidth(1.8 if fresh else 0.8)
            if steps[i]["kind"] == "circle":
                art.set_edgecolor(ACCENT[1] if fresh else ACCENT[0])
            else:
                art.set_color(ACCENT[1] if fresh else "#374151")
        for name, dot in dots.items():
            dot.set_alpha(0.9 if u > born[name] + 1 else 0)
        for name, tag in tags.items():
            tag.set_alpha(1 if u > born[name] + 1 else 0)
        shown = [k for k in range(17) if u > born.get(f"P{k}", -1) + 1]
        vdots.set_alpha(1 if shown else 0)
        vdots.set_offsets(verts[shown] if shown else np.empty((0, 2)))
        k_step = min(n_steps - 1, int(u))
        if f < f_build:
            title_c.set_text(f"step {k_step + 1}/{n_steps}: {steps[k_step]['label']}")
        else:
            g = (f - f_build) / POLY_FRAMES
            m = min(17, round(17 * g)) + 1
            edges.set_data(closed[:m, 0], closed[:m, 1])
            fill.set_alpha(0.25 * min(1.0, max(0.0, 2 * g - 1)))
            title_c.set_text("the regular 17-gon, Richmond (1893)")
            foot.set_text(f"{n_steps} ruler-and-compass steps, max vertex error {err:.1e} ({d['precision']})")
        for m in range(5):
            for art in tree_art[m]:
                art.set_alpha(1 if frac >= m / 5 else 0)
        for m, art in enumerate(rad_art):
            art.set_alpha(1 if frac >= (m + 1) / 5 else 0)
        if frac >= 1:
            gauss_txt.set_text(f"= {16 * d['gauss']:.13f}, cos error {abs(d['gauss'] - np.cos(2 * np.pi / 17)):.0e}")
        reveal = int(round(n_max * min(1.0, f / (0.8 * f_build))))
        full[:] = 0
        full[:reveal] = code[:reveal]
        im.set_data(full.reshape(rows, cols))
        for idx, t in enumerate(texts):
            t.set_alpha(1 if idx < reveal else 0)

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(n_frames, n_frames), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'generator': '3',
    'precision': 'float64',
    'n_max': 100
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.gauss_heptadecagon")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.gauss_heptadecagon.main")
problem_dir = Path("problems/maths/gauss_heptadecagon")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
